In [ ]:
!pip install pandas tqdm scikit-learn matplotlib opencv-python grad-cam

In [ ]:
# Get training and validation CSVs
!wget -nc "https://filebrowser-oogwckc88o8kgowgcow80kos.vaffel.org/api/public/dl/C5dRKoAs/home/train_visualCheXbert.csv"
!wget -nc "https://filebrowser-oogwckc88o8kgowgcow80kos.vaffel.org/api/public/dl/CdTLaiED/home/valid.csv"

In [ ]:
# Run if dataset is not already unzipped in DATA_DIR
!wget -nc -O chexpert.zip https://www.kaggle.com/api/v1/datasets/download/ashery/chexpert
print("Unzipping archive... This should take 2-3 minutes in Colab.")
!unzip -q chexpert.zip -d CheXpert-v1.0-small
print("Done.")

## Imports

In [ ]:
import os, numpy as np, pandas as pd
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet50_Weights, ResNet152_Weights, DenseNet121_Weights, ResNeXt50_32X4D_Weights
from PIL import Image
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau, LinearLR
from sklearn.metrics import roc_auc_score, recall_score
# Grad-CAM imports
import matplotlib.pyplot as plt
import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    print(f"torch_xla version: {torch_xla.__version__}")
    XLA_AVAILABLE = True
except ImportError:
    print("torch_xla not found. TPU support disabled.")
    XLA_AVAILABLE = False

# Device Selection Logic
if XLA_AVAILABLE:
    # Gets the assigned TPU core device
    DEVICE = xm.xla_device()
    DEVICE_TYPE = 'xla'
    print(f"TPU available! Using device: {DEVICE}")
    print(f"World size (number of TPU cores): {xm.torch_xla.runtime.world_size()}")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DEVICE_TYPE = 'cuda'
    print(f"GPU available! Using device: {DEVICE}")
else:
    DEVICE = torch.device("cpu")
    DEVICE_TYPE = 'cpu'
    print(f"TPU/GPU not available. Using device: {DEVICE}")

# Import Drive only if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab environment.")
except ImportError:
    IN_COLAB = False
    print("Not running in Google Colab environment.")

if IN_COLAB:
    from google.colab import drive

## Configuration

In [ ]:
DATA_DIR = '/content'
MODEL_NAME = 'ResNet50' # Options: 'ResNet50', 'ResNet152' 'DenseNet121', 'ResNeXt50_32x4d'

# Adjust batch size according to GPU memory and model.
BATCH_SIZE = 64
NUM_EPOCHS = 50

# Optimizer
LR = 5e-5 # Learning rate
LR_FACTOR = 0.1 # Factor by which the learning rate will be reduced. new_lr = lr * factor
LR_PATIENCE = 3 # Number of epochs with no improvement after which learning rate will be reduced.
MIN_LR = 1e-6   # A lower bound on the learning rate
WD = 1e-5 # Define weight decay value
WARMUP_EPOCHS = 1 # Number of epochs for the warmup scheduler

EARLY_STOPPING_PATIENCE = 7 # Number of epochs to wait for improvement before stopping

QUICK_TEST_MODE = False  # Set to True to run only a few batches, False for full training. Useful for debugging.
QUICK_TEST_BATCHES = 10 # Number of batches to run in train and validation loops during quick test

# Use num_workers > 0 for background loading, pin_memory conditionally
# Set num_workers=0 if you encounter issues, especially on Windows or in certain environments
# Colab seems to require 2
num_dataloader_workers = 2

# Google Drive Configuration
USE_GOOGLE_DRIVE_FOR_CHECKPOINTS = True # Set to True to use Drive when in Colab
DRIVE_MOUNT_POINT = '/content/drive'
# Path within Drive
DRIVE_PROJECT_FOLDER = 'MyDrive/dat255/training'

## Label definition

In [ ]:
LABELS = ['No Finding',
          'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion',
          'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax',
          'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']
NUM_CLASSES = len(LABELS)


## Load data

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_visualCheXbert.csv'))
valid_df = pd.read_csv(os.path.join(DATA_DIR, 'valid.csv'))

# Process training labels (VisualCheXbert)
print("Processing training labels (VisualCheXbert)...")
# Select columns by name in the desired order and fill NaNs
train_targets_df = train_df[LABELS].fillna(0.0)
# Convert to float32 numpy array
train_targets = train_targets_df.values.astype(np.float32)

# Process validation labels
print("Processing validation labels...")
# Select columns by name in the desired order and fill NaNs
valid_targets_df = valid_df[LABELS].fillna(0.0)
# Convert to float32 numpy array
valid_targets = valid_targets_df.values.astype(np.float32)

# Verify shapes
print("Train targets shape:", train_targets.shape)
print("Valid targets shape:", valid_targets.shape)
if train_targets.shape[1] != NUM_CLASSES or valid_targets.shape[1] != NUM_CLASSES:
    raise ValueError("Mismatch between number of labels extracted and NUM_CLASSES")

print("Label processing finished.")

## Define dataset class

In [ ]:
class CheXpertDataset(Dataset):
    def __init__(self, df, targets, transform=None):
        self.paths = df['Path'].values
        self.full_paths = [os.path.join(DATA_DIR, p) for p in self.paths]
        self.targets = torch.FloatTensor(targets)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.full_paths[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            target = self.targets[idx]
            return image, target
        except FileNotFoundError:
            print(f"Warning: File not found {img_path}. Skipping.")
            # Return None or raise an error, or handle appropriately
            print("Returning dummy data for missing file to avoid crash.")
            img_path_0 = self.full_paths[0]
            image = Image.open(img_path_0).convert('RGB')
            if self.transform:
                image = self.transform(image)
            target = self.targets[0]
            return image, target

train_transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), shear=5),
    transforms.RandAugment(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
valid_transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds = CheXpertDataset(train_df, train_targets, train_transform)
valid_ds = CheXpertDataset(valid_df, valid_targets, valid_transform)

# Determine pin_memory setting based on device type
use_pin_memory = (DEVICE_TYPE == 'cuda') # Set to True only if using GPU
print(f"Using pin_memory = {use_pin_memory} for DataLoaders.")

# Initialize loader variables
train_loader = None
valid_loader = None
train_loader_device = None
valid_loader_device = None
num_train_batches = 0

# TPU initialization
if DEVICE_TYPE == 'xla':
    print("Initializing DataLoaders for TPU...")
    _train_loader = DataLoader(train_ds,
                               batch_size=BATCH_SIZE,
                               shuffle=True,
                               num_workers=num_dataloader_workers,
                               pin_memory=use_pin_memory,
                               drop_last=True) # Ensure even batch sizes per core

    _valid_loader = DataLoader(valid_ds,
                               batch_size=BATCH_SIZE,
                               shuffle=False,
                               num_workers=num_dataloader_workers,
                               pin_memory=use_pin_memory,
                               drop_last=True) # Ensure even batch sizes per core

    train_loader = pl.ParallelLoader(_train_loader, [DEVICE])
    valid_loader = pl.ParallelLoader(_valid_loader, [DEVICE])

    # Get the actual iterator for the training loop
    train_loader_device = train_loader.per_device_loader(DEVICE)
    valid_loader_device = valid_loader.per_device_loader(DEVICE)
    print("DataLoaders wrapped with ParallelLoader.")
    num_train_batches = len(_train_loader)

# GPU or CPU initialization
else:
    train_loader = DataLoader(train_ds,
                              batch_size=BATCH_SIZE,
                              shuffle=True,
                              num_workers=num_dataloader_workers,
                              pin_memory=use_pin_memory)

    valid_loader = DataLoader(valid_ds,
                              batch_size=BATCH_SIZE,
                              shuffle=False,
                              num_workers=num_dataloader_workers,
                              pin_memory=use_pin_memory)

    # For consistency in the training loop, assign the loaders directly
    train_loader_device = train_loader
    valid_loader_device = valid_loader
    print("Using standard DataLoaders.")
    num_train_batches = len(train_loader)

## Model definition

In [ ]:
print(f"Initializing model: {MODEL_NAME}")

if MODEL_NAME == 'ResNet50':
    model = models.resnet50(weights=ResNet50_Weights.DEFAULT)
    in_features = model.fc.in_features
    dropout_prob = 0.3
    print(f"Adding Dropout layer with p={dropout_prob}")
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout_prob),
        nn.Linear(in_features, NUM_CLASSES)
    )

elif MODEL_NAME == 'ResNet152':
    model = models.resnet152(weights=ResNet152_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, NUM_CLASSES)

elif MODEL_NAME == 'DenseNet121':
    model = models.densenet121(weights=DenseNet121_Weights.DEFAULT)
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, NUM_CLASSES)

elif MODEL_NAME == 'ResNeXt50_32x4d':
    model = models.resnext50_32x4d(weights=ResNeXt50_32X4D_Weights.DEFAULT)
    in_features = model.fc.in_features
    dropout_prob = 0.3
    print(f"Adding Dropout layer with p={dropout_prob}")
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout_prob),
        nn.Linear(in_features, NUM_CLASSES)
    )

else:
    raise ValueError(f"Unsupported MODEL_NAME: {MODEL_NAME}. Choose from 'ResNet50', 'ResNet152', 'DenseNet121', 'ResNeXt50_32x4d'")

model = model.to(DEVICE)
print(f"Model {MODEL_NAME} initialized and moved to {DEVICE}.")

criterion = nn.BCEWithLogitsLoss()  # multi-label

# print(f"Using Adam optimizer with LR={LR}")
# optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(f"Using AdamW optimizer with LR={LR}, weight_decay={WD}")
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

# Monitor validation AUC and reduce LR if it doesn't improve
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=MIN_LR)

# Warmup scheduler (for the first WARMUP_EPOCHS epoch(s))
warmup_total_iters = num_train_batches * WARMUP_EPOCHS
print(f"Initializing LinearLR warmup scheduler for {WARMUP_EPOCHS} epoch(s) ({warmup_total_iters} iterations)...")
warmup_scheduler = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_total_iters)

# Mount Google Drive if in Colab and requested
DRIVE_CHECKPOINT_DIR = None
if IN_COLAB and USE_GOOGLE_DRIVE_FOR_CHECKPOINTS:
    print(f"Attempting to mount Google Drive at: {DRIVE_MOUNT_POINT}")
    try:
        drive.mount(DRIVE_MOUNT_POINT, force_remount=True) # force_remount can help if needed
        DRIVE_CHECKPOINT_DIR = os.path.join(DRIVE_MOUNT_POINT, DRIVE_PROJECT_FOLDER)
        print(f"Google Drive mounted. Checkpoint directory set to: {DRIVE_CHECKPOINT_DIR}")
        # Create the directory structure in Drive if it doesn't exist
        os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
        print("Checkpoint directory checked/created.")
    except Exception as e:
        print(f"Error mounting Google Drive: {e}")
        print("Checkpoints will be saved locally in the Colab runtime.")
        # Fallback to local storage if mounting fails
        DRIVE_CHECKPOINT_DIR = None
        USE_GOOGLE_DRIVE_FOR_CHECKPOINTS = False # Disable drive usage if mount failed
else:
    if IN_COLAB:
        print("Google Drive usage for checkpoints is disabled by configuration.")
    else:
        print("Not in Colab, checkpoints will be saved/loaded locally.")

## Training definition

In [ ]:
### 4. Training Loop
def train_epoch(model, loader, current_epoch):
    model.train()
    running_loss = 0.0

    if QUICK_TEST_MODE:
        print(f"QUICK TEST MODE: Running only {QUICK_TEST_BATCHES} training batches")

    actual_loader = train_loader_device if DEVICE_TYPE == 'xla' else loader

    num_samples_processed = 0

    for batch_idx, (images, targets) in enumerate(tqdm(actual_loader, desc="Training")):
        # Data automatically moved to DEVICE by ParallelLoader if using TPU
        # If using GPU/CPU, move data manually
        if DEVICE_TYPE != 'xla':
             images, targets = images.to(DEVICE), targets.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()

        if DEVICE_TYPE == 'xla':
            # xm.optimizer_step synchronizes gradients and updates weights across TPU cores
            xm.optimizer_step(optimizer)
        else:
            # Standard optimizer step for CPU/GPU
            optimizer.step()

        if current_epoch < WARMUP_EPOCHS:
             warmup_scheduler.step()

        batch_size = images.size(0) # Get actual batch size
        running_loss += loss.item() * batch_size
        num_samples_processed += batch_size

        if QUICK_TEST_MODE and batch_idx >= (QUICK_TEST_BATCHES - 1): # batch_idx starts at 0
            print(f"\nQuick test: Reached {QUICK_TEST_BATCHES} batches, stopping train epoch early.")
            break

    epoch_loss = running_loss / num_samples_processed if num_samples_processed > 0 else 0.0

    if QUICK_TEST_MODE:
         print(f"Quick test: Calculated loss based on {num_samples_processed} samples.")
    else:
      pass

    return epoch_loss

## Validation definition

In [ ]:
def validate(model, loader):
    model.eval()
    all_preds = []
    all_gts = []
    num_samples_processed = 0

    if QUICK_TEST_MODE:
        print(f"QUICK TEST MODE: Running only {QUICK_TEST_BATCHES} validation batches")

    actual_loader = valid_loader_device if DEVICE_TYPE == 'xla' else loader

    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(tqdm(actual_loader, desc="Validating")):
            if DEVICE_TYPE != 'xla':
                 images = images.to(DEVICE)

            gts_np = targets.cpu().numpy() # Move targets to CPU for numpy conversion
            num_samples_processed += images.size(0) # Count samples based on image batch size

            logits = model(images)
            probs = torch.sigmoid(logits).cpu().numpy() # Move predictions to CPU for numpy/sklearn

            all_preds.append(probs)
            all_gts.append(gts_np)

            if QUICK_TEST_MODE and batch_idx >= (QUICK_TEST_BATCHES - 1): # batch_idx starts at 0
                 print(f"\nQuick test: Reached {QUICK_TEST_BATCHES} batches, stopping validation early.")
                 break

    # Ensure arrays are not empty before concatenating, especially if QUICK_TEST_BATCHES is small
    if not all_preds or not all_gts:
        print("Warning: No batches processed during validation quick test. Returning dummy metrics.")
        return 0.5, 0.0

    preds = np.concatenate(all_preds)
    gts = np.concatenate(all_gts)

    aucs, recalls = [], []
    print("\nValidation Metrics per Class:")
    for i, label_name in enumerate(LABELS):
        try:
            # Ensure there are both classes present in gts for this label
            if len(np.unique(gts[:, i])) > 1:
                auc = roc_auc_score(gts[:, i], preds[:, i])
            else:
                print(f"Skipping AUC for {label_name}: Only one class present in validation ground truth.")
                auc = np.nan
        except ValueError as e:
            print(f"Could not calculate AUC for {label_name}: {e}")
            auc = np.nan

        # Calculate recall with a 0.5 threshold for monitoring
        preds_binary = (preds[:, i] >= 0.5).astype(int)
        recall = recall_score(gts[:, i], preds_binary, zero_division=0, average='binary', pos_label=1)

        aucs.append(auc)
        recalls.append(recall)
        print(f"  {label_name:<25}: AUC = {auc:.4f}, Recall@0.5 = {recall:.4f}")

    mean_auc = np.nanmean(aucs)
    mean_recall = np.nanmean(recalls) # Use nanmean in case some recalls were NaN

    if QUICK_TEST_MODE:
        print(f"Quick test: Calculated validation metrics based on {num_samples_processed} samples.")

    print(f"\nOverall Mean AUC: {mean_auc:.4f}, Mean Recall@0.5: {mean_recall:.4f}")
    return mean_auc, mean_recall


## Grad-CAM helper function definitions

In [ ]:
def get_target_layer(model, model_name):
    """Selects the last convolutional layer suitable for Grad-CAM."""
    if model_name.startswith('ResNet') or model_name.startswith('ResNeXt'):
        return model.layer4[-1] # Last block of the last layer
    elif model_name.startswith('DenseNet'):
        return model.features # Last feature block
    else:
        print(f"Warning: Target layer not defined for model: {model_name}. Using generic last conv if possible.")
        return None

# Define a transform specifically for CAM visualization input
cam_transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor(), # Convert to [0, 1] range first
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]), # Then normalize
])

def prepare_image_for_cam(img_path, transform):
    """Loads image, returns tensor for model and np array for viz."""
    try:
        img_pil = Image.open(img_path).convert('RGB')
        # For visualization: resize and convert to np array [0, 1]
        img_for_viz = np.array(img_pil.resize((320, 320))) / 255.0
        # For model input: apply the full transform
        input_tensor = transform(img_pil).unsqueeze(0).to(DEVICE)
        return input_tensor, img_for_viz
    except Exception as e:
        print(f"Error processing image {img_path} for CAM: {e}")
        return None, None

def generate_and_display_cam(cam_runner, model, model_name, target_layer, img_path, target_class_index, transform):
    """Generates and displays Grad-CAM using matplotlib."""
    input_tensor, rgb_img_for_viz = prepare_image_for_cam(img_path, transform)
    if input_tensor is None:
        return

    # Set target layer for the CAM runner instance, ensure it's a list
    cam_runner.target_layers = [target_layer]

    # Define targets for CAM generation
    targets = [ClassifierOutputTarget(target_class_index)]

    # Generate CAM
    grayscale_cam = cam_runner(input_tensor=input_tensor, targets=targets)
    grayscale_cam = grayscale_cam[0, :] # Get first heatmap

    # Create overlay
    visualization = show_cam_on_image(rgb_img_for_viz, grayscale_cam, use_rgb=True)

    # Display
    target_class_name = LABELS[target_class_index]
    # Get probability for display title
    logits = model(input_tensor)
    probability = torch.sigmoid(logits).squeeze()[target_class_index].item()


    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(rgb_img_for_viz)
    plt.title(f'Original Image\n({os.path.basename(img_path)})')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(visualization)
    plt.title(f'{type(cam_runner).__name__} for: {target_class_name}\nProb: {probability:.3f}')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## Preparing for training

In [ ]:
# Initialise metric trackers for plotting
train_losses = []
val_aucs = []
val_recalls = []

start_epoch = 0
best_val_auc = 0.0
epochs_no_improve = 0

# Checkpoint loading logic
checkpoint_filename = f'best_checkpoint_{MODEL_NAME}.pth'

if IN_COLAB and USE_GOOGLE_DRIVE_FOR_CHECKPOINTS and DRIVE_CHECKPOINT_DIR:
    # Use the path inside Google Drive
    resume_path = os.path.join(DRIVE_CHECKPOINT_DIR, checkpoint_filename)
    print(f"Checkpoint path set to Google Drive: {resume_path}")
else:
    # Use local path (current directory or Colab runtime storage)
    resume_path = checkpoint_filename
    print(f"Checkpoint path set to local runtime: {resume_path}")

if os.path.isfile(resume_path):
    print(f"Resuming from {resume_path}")

    print("Loading checkpoint to CPU...")
    # Load the entire checkpoint dictionary onto the CPU memory (required for TPU)
    checkpoint = torch.load(resume_path, map_location='cpu', weights_only=False)
    print("Checkpoint loaded to CPU.")

    # Load model state dict
    print("Loading model state dict...")
    model_state_dict = checkpoint['model_state_dict']
    model.load_state_dict(model_state_dict) # load_state_dict handles moving params to model's device
    print("Model state dict loaded.")

    # Load optimizer state dict
    print("Loading optimizer state dict...")
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print("Optimizer state dict loaded.")

    # Load other checkpoint info
    start_epoch = checkpoint['epoch'] + 1
    best_val_auc = checkpoint.get('best_val_auc', 0.0)

    # Load scheduler state
    if 'scheduler_state_dict' in checkpoint:
        print("Loading scheduler state dict...")
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        print("Scheduler state dict loaded.")
    else:
        print("Scheduler state not found in checkpoint.")

    if 'warmup_scheduler_state_dict' in checkpoint:
        warmup_scheduler.load_state_dict(checkpoint['warmup_scheduler_state_dict'])
        print("Loaded warmup_scheduler state.")
    else:
        print("Warmup scheduler state not found in checkpoint.")

    # Load metric history
    train_losses = checkpoint.get('train_losses', [])
    val_aucs = checkpoint.get('val_aucs', [])
    val_recalls = checkpoint.get('val_recalls', [])

    # Calculate epochs_no_improve
    last_best_epoch = checkpoint.get('epoch', -1)
    if last_best_epoch != -1:
         epochs_no_improve = (start_epoch - 1) - last_best_epoch
         epochs_no_improve = max(0, epochs_no_improve)
         print(f"Resuming: {epochs_no_improve} epochs since last validation AUC improvement (at epoch {last_best_epoch + 1}).")
    else:
         epochs_no_improve = 0
         print("Warning: Could not determine epochs since last improvement. Resetting counter.")

    print(f"Resuming training from Epoch {start_epoch}")

else:
    print("Starting training from scratch")

## Train the model

In [ ]:
print(f"Starting training loop for up to {NUM_EPOCHS} epochs...")
print(f"Warmup active for first {WARMUP_EPOCHS} epoch(s).")
print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE} epochs.")
for epoch in range(start_epoch, NUM_EPOCHS):
    current_lr = optimizer.param_groups[0]['lr']
    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    print(f"Current LR: {current_lr:.6f}")

    train_loss = train_epoch(model, train_loader, epoch)
    val_auc, val_recall = validate(model, valid_loader)

    train_losses.append(train_loss)
    val_aucs.append(val_auc)
    val_recalls.append(val_recall)

    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val AUC:    {val_auc:.4f} (Best: {best_val_auc:.4f})")
    print(f"  Val Recall: {val_recall:.4f}")

    scheduler.step(val_auc)

    lr_after_plateau_step = optimizer.param_groups[0]['lr']
    if lr_after_plateau_step < current_lr: # Compare with LR at start of epoch
         print(f"  ReduceLROnPlateau reduced learning rate: {current_lr:.6f} -> {lr_after_plateau_step:.6f}")

    # Save checkpoint logic - use the current epoch number
    current_epoch_val_auc = val_auc

    if current_epoch_val_auc > best_val_auc:
        print(f"Validation AUC improved ({best_val_auc:.4f} --> {current_epoch_val_auc:.4f}). Saving checkpoint...")
        best_val_auc = current_epoch_val_auc

        # Create the checkpoint dictionary
        checkpoint_data = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'warmup_scheduler_state_dict': warmup_scheduler.state_dict(),
            'val_auc': current_epoch_val_auc,
            'best_val_auc': best_val_auc,
            'train_losses': train_losses,
            'val_aucs': val_aucs,
            'val_recalls': val_recalls,
        }

        if DEVICE_TYPE == 'xla':
             xm.save(checkpoint_data, resume_path)
             print(f"Checkpoint saved to {resume_path}")
        else:
             torch.save(checkpoint_data, resume_path)
             print(f"Checkpoint saved to {resume_path}")
    else:
        epochs_no_improve += 1
        print(f"Validation AUC did not improve from {best_val_auc:.4f}.")
        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping triggered after {epochs_no_improve} epochs without improvement.")
            break

print("Training finished.")

## Inspecting results

In [ ]:
import matplotlib.pyplot as plt

# Ensure epoch_axis aligns with the actual number of epochs run/logged
actual_epochs_run = len(train_losses) # Should match the number of completed epochs
epoch_axis = range(start_epoch - len(train_losses) + 1, start_epoch + 1) if os.path.isfile(resume_path) and train_losses else range(1, actual_epochs_run + 1)
epoch_axis = range(1, actual_epochs_run + 1)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(epoch_axis, train_losses, 'b-o', label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss over Epochs')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epoch_axis, val_aucs, 'r-o', label='Validation AUC')
plt.plot(epoch_axis, val_recalls, 'g-s', label='Validation Recall (Thr=0.5)')
plt.xlabel('Epoch')
plt.ylabel('Metric Value')
plt.title('Validation Metrics over Epochs')
# Add marker for best AUC epoch
if val_aucs:
    best_epoch_idx = np.argmax(val_aucs)
    plt.plot(epoch_axis[best_epoch_idx], val_aucs[best_epoch_idx], 'r*', markersize=15, label=f'Best AUC ({val_aucs[best_epoch_idx]:.4f})')

plt.legend()
plt.grid(True)
plt.ylim(bottom=0)

plt.tight_layout()

plot_filename = f'training_curves_{MODEL_NAME}.png'

if IN_COLAB and USE_GOOGLE_DRIVE_FOR_CHECKPOINTS and DRIVE_CHECKPOINT_DIR:
    plot_save_path = os.path.join(DRIVE_CHECKPOINT_DIR, plot_filename)
else:
    plot_save_path = plot_filename

try:
    plt.savefig(plot_save_path)
    print(f"Saved training curves plot to: {plot_save_path}")
except Exception as e:
    print(f"Error saving plot to {plot_save_path}: {e}")

plt.show()

## Testing Grad-CAM

In [ ]:
print("\nGenerating Grad-CAM Visualizations")

# Load the best model state from the checkpoint
# Ensure the model variable holds the architecture used for the best checkpoint
print(f"Loading best model state for {MODEL_NAME} from: {resume_path}")
if os.path.isfile(resume_path):
    try:
        checkpoint = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        # Load the state dict into the existing model architecture
        state_dict = checkpoint['model_state_dict']
        new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        model.load_state_dict(new_state_dict)
        model.eval()
        best_epoch_auc = checkpoint.get('best_val_auc', 'N/A')
        print(f"Successfully loaded best model weights (Epoch {checkpoint.get('epoch', 'N/A')+1}, Val AUC: {best_epoch_auc:.4f}).")

        # Prepare Grad-CAM
        target_layer_for_cam = get_target_layer(model, MODEL_NAME)
        if target_layer_for_cam:
            print("Initializing GradCAM runner...")
            cam_runner = GradCAM(model=model, target_layers=[target_layer_for_cam])

            print("GradCAM runner initialized.")

            # Images and classes to visualize
            num_viz_images = 5
            classes_to_visualize = ['Cardiomegaly', 'Edema', 'Pneumonia', 'Atelectasis', 'Pleural Effusion']

            print(f"\nGenerating CAMs for {num_viz_images} validation images and classes: {classes_to_visualize}")

            # Get some samples from the validation dataset
            viz_indices = np.random.choice(len(valid_ds), num_viz_images, replace=False)

            for img_idx in viz_indices:
                # Get image path from validation dataset
                image_path = valid_ds.full_paths[img_idx]
                print(f"\nVisualizing image: {image_path}")

                # Get ground truth for context
                gt_labels = valid_ds.targets[img_idx].numpy()
                gt_positive_indices = np.where(gt_labels == 1)[0]
                gt_positive_labels = [LABELS[i] for i in gt_positive_indices]
                print(f"  Ground Truth: {gt_positive_labels if gt_positive_labels else 'No Findings'}")

                # Generate CAM for specified classes
                for class_name in classes_to_visualize:
                    if class_name in LABELS:
                        target_class_index = LABELS.index(class_name)
                        try:
                           generate_and_display_cam(cam_runner, model, MODEL_NAME, target_layer_for_cam, image_path, target_class_index, cam_transform)
                        except Exception as e_cam:
                            print(f"  Error generating CAM for {class_name}: {e_cam}")
                    else:
                        print(f"  Skipping visualization for unknown class: {class_name}")
        else:
             print("Could not identify target layer for CAM generation.")

    except Exception as e:
        print(f"Could not load best model or generate CAMs: {e}")
else:
    print(f"Best checkpoint not found at {resume_path}. Skipping Grad-CAM generation.")